# SEED 1

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.8 MB/s eta 0:00:00


In [ ]:
import os, shutil, zipfile
from google.colab import drive

# 1. Mount Drive (nếu chưa mount)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# --- CẤU HÌNH ĐƯỜNG DẪN (Kiểm tra kỹ tên file trên Drive của bạn) ---
DRIVE_FOLDER = "/content/drive/MyDrive/TechNo Project/"
IMG_ZIP_PATH = "/content/drive/MyDrive/TechNo Project/dataset_full.zip" # Đường dẫn tới file chứa 4000+ ảnh
LABEL_ZIPS = {
    'head_shoulder_top': 0,
    'head_shoulder_bottom': 1,
    'double_top': 2,
    'double_bottom': 3
}

# Thư mục làm việc trên Colab
RAW_IMG_DIR = "/content/raw_images"
ROOT_DATA = "/content/unified_data"
os.makedirs(f"{ROOT_DATA}/labels", exist_ok=True)
os.makedirs(f"{ROOT_DATA}/images", exist_ok=True)

# 2. Giải nén kho ảnh khổng lồ (4000+ ảnh)
if not os.path.exists(RAW_IMG_DIR):
    print(" đang giải nén kho ảnh gốc (dataset_full.zip)...")
    with zipfile.ZipFile(IMG_ZIP_PATH, 'r') as z:
        z.extractall(RAW_IMG_DIR)
else:
    print(" Thư mục ảnh gốc đã tồn tại, bỏ qua bước giải nén ảnh.")

# 3. Xử lý từng file nhãn và khớp với ảnh
total_images_found = 0

for zip_name, new_id in LABEL_ZIPS.items():
    zip_full_path = os.path.join(DRIVE_FOLDER, f"{zip_name}.zip")

    if not os.path.exists(zip_full_path):
        print(f"⚠️ Cảnh báo: Không tìm thấy file {zip_full_path}")
        continue

    temp_extract = f"/content/temp_{zip_name}"
    with zipfile.ZipFile(zip_full_path, 'r') as z:
        z.extractall(temp_extract)

    print(f" Đang xử lý nhãn từ: {zip_name} (ID: {new_id})")

    # Duyệt qua các file .txt nhãn vừa giải nén
    for root, dirs, files in os.walk(temp_extract):
        for txt_file in files:
            if txt_file.endswith('.txt') and not txt_file.startswith('classes'):
                # Đọc nhãn cũ và cập nhật ID lớp
                with open(os.path.join(root, txt_file), 'r') as f:
                    lines = f.readlines()

                # Lưu nhãn mới vào thư mục unified_data
                target_txt_path = os.path.join(ROOT_DATA, "labels", txt_file)
                with open(target_txt_path, 'a') as f: # Dùng 'a' để hỗ trợ ảnh có nhiều nhãn
                    for line in lines:
                        parts = line.split()
                        if len(parts) > 0:
                            parts[0] = str(new_id) # Cập nhật Class ID
                            f.write(" ".join(parts) + "\n")

                # Tìm ảnh tương ứng trong kho RAW_IMG_DIR
                img_name = txt_file.replace('.txt', '.png')
                # Tìm kiếm sâu trong các thư mục con của RAW_IMG_DIR nếu có
                found_img_path = None
                for r_img, d_img, f_img in os.walk(RAW_IMG_DIR):
                    if img_name in f_img:
                        found_img_path = os.path.join(r_img, img_name)
                        break

                if found_img_path:
                    shutil.copy(found_img_path, os.path.join(ROOT_DATA, "images", img_name))
                    total_images_found += 1
                else:
                    # Nếu không tìm thấy ảnh, xóa file nhãn vừa tạo để tránh rác
                    if os.path.exists(target_txt_path):
                        os.remove(target_txt_path)

# Dọn dẹp thư mục tạm
for zip_name in LABEL_ZIPS.keys():
    shutil.rmtree(f"/content/temp_{zip_name}", ignore_errors=True)

print(f"\n✅ THÀNH CÔNG!")
print(f"- Tổng số cặp Ảnh-Nhãn hợp lệ thu thập được: {len(os.listdir(ROOT_DATA+'/images'))}")
print(f"- Dữ liệu đã sẵn sàng tại: {ROOT_DATA}")

 đang giải nén kho ảnh gốc (dataset_full.zip)...
 Đang xử lý nhãn từ: head_shoulder_top (ID: 0)
 Đang xử lý nhãn từ: head_shoulder_bottom (ID: 1)
 Đang xử lý nhãn từ: double_top (ID: 2)
 Đang xử lý nhãn từ: double_bottom (ID: 3)

✅ THÀNH CÔNG!
- Tổng số cặp Ảnh-Nhãn hợp lệ thu thập được: 253
- Dữ liệu đã sẵn sàng tại: /content/unified_data


In [ ]:
import os, random, shutil
from pathlib import Path

ROOT = Path("/content/unified_data")
imgs = list((ROOT/"images").glob("*.png"))
random.shuffle(imgs)

split = int(0.8*len(imgs))
train, val = imgs[:split], imgs[split:]

for p in ["train/images","train/labels","val/images","val/labels"]:
    (ROOT/p).mkdir(parents=True, exist_ok=True)

for s, subset in [(train,"train"),(val,"val")]:
    for img in s:
        lbl = ROOT/"labels"/(img.stem+".txt")
        shutil.copy(img, ROOT/f"{subset}/images")
        shutil.copy(lbl, ROOT/f"{subset}/labels")

yaml = """
path: /content/unified_data
train: train/images
val: val/images
names:
  0: HSA
  1: RHSA
  2: DT
  3: DB
"""
open("seed.yaml","w").write(yaml)

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="seed.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    cache=True,
    amp=True,
    device=0,
    project="/content/drive/MyDrive/TechNo_Project/Model_Seed",
    name="Seed_v1"
)



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=seed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f53dc2dd670>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
import shutil
from glob import glob
from tqdm import tqdm

SAVE_PATH = "/content/drive/MyDrive/TechNo_Project/Model_Seed"

model = YOLO(f"{SAVE_PATH}/Seed_v1/weights/best.pt")
RAW_IMAGES = "/content/raw_images" # Thư mục 4000 ảnh
PSEUDO_DIR = "/content/pseudo_data"
os.makedirs(f"{PSEUDO_DIR}/images", exist_ok=True)
os.makedirs(f"{PSEUDO_DIR}/labels", exist_ok=True)

raw_files = glob(f"{RAW_IMAGES}/**/*.png", recursive=True)
for img_p in tqdm(raw_files[:4600]): # Quét 3000 ảnh
    res = model.predict(img_p, conf=0.65, verbose=False) # Chỉ lấy mẫu cực chuẩn
    if len(res[0].boxes) > 0:
        name = os.path.basename(img_p)
        shutil.copy(img_p, f"{PSEUDO_DIR}/images/{name}")
        res[0].save_txt(f"{PSEUDO_DIR}/labels/{name.replace('.png', '.txt')}")

print(f"\nAI đã săn thêm được {len(os.listdir(PSEUDO_DIR+'/images'))} mẫu hình!")

100%|██████████| 4600/4600 [00:52<00:00, 87.41it/s]


AI đã săn thêm được 426 mẫu hình!


# Augmentation

In [ ]:
# ======= Augmentation an toàn cho chart + volume (thay phần augmentation cũ) =======
import os, cv2, random, math, shutil
from glob import glob
from pathlib import Path
from tqdm import tqdm
import albumentations as A
import numpy as np

# --- CẤU HÌNH TÙY CHỈNH ---
FINAL_DIR = "/content/final_dataset"   # nơi lưu train/val (bạn đã tạo trước đó)
GOLD_DIR = "/content/unified_data"
PSEUDO_DIR = "/content/pseudo_data"

# Mục tiêu khoảng bao nhiêu ảnh train cuối cùng (không bắt buộc)
TARGET_TOTAL_TRAIN = 9000

# Tối đa tạo thêm bao nhiêu augment trên 1 ảnh gốc
MAX_AUG_PER_IMAGE = 20

# Ngưỡng chấp nhận bbox sau augment (so với area gốc):
# nếu diện tích bounding box giảm < MIN_AREA_RATIO*original_area thì loại augment
MIN_AREA_RATIO = 0.85

# Ngưỡng phần bbox phải nằm trong ảnh (tỉ lệ phần bbox không bị crop)
MIN_IOU_INSIDE = 0.98  # 98% bbox phải còn trong image

# Số lần thử cho mỗi ảnh để tạo augment hợp lệ
MAX_TRIES_PER_IMAGE = 40

# Tỉ lệ giữ ảnh gốc (không augment) trong train
KEEP_ORIGINAL = True

# --- AUGMENTATIONS (chỉ các phép an toàn) ---
# Không dùng RandomResizedCrop, No flips.
safe_transform = A.Compose([
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=0.7),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.3)
    ], p=0.8),
    A.OneOf([
        A.GaussNoise(var_limit=(5.0, 25.0), p=0.5),
        A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1,0.5), p=0.25)
    ], p=0.6),
    A.OneOf([
        A.MotionBlur(blur_limit=3, p=0.25),
        A.MedianBlur(blur_limit=3, p=0.15),
        A.GaussianBlur(blur_limit=3, p=0.15)
    ], p=0.45),
    # Small geometric transforms: rotate/scale/shift very tiny and KEEP border reflect
    A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.03, rotate_limit=2, border_mode=cv2.BORDER_REFLECT, p=0.6),
    # small elastic distortions (soft)
    A.OneOf([
        A.GridDistortion(num_steps=4, distort_limit=0.02, p=0.2),
        A.OpticalDistortion(distort_limit=0.02, shift_limit=0.01, p=0.15)
    ], p=0.25),
    # minor color jitter in value space (not hue)
    A.HueSaturationValue(hue_shift_limit=0, sat_shift_limit=8, val_shift_limit=15, p=0.25),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

# --- Utility functions ---
def read_yolo_label(lbl_path):
    boxes = []
    classes = []
    if not os.path.exists(lbl_path):
        return boxes, classes
    for ln in open(lbl_path):
        parts = ln.strip().split()
        if len(parts) >= 5:
            classes.append(int(parts[0]))
            boxes.append(list(map(float, parts[1:5])))
    return boxes, classes

def yolo_area(box):
    # box = [cx,cy,w,h] normalized -> area fraction
    return float(box[2]) * float(box[3])

def bbox_inside_fraction(yolo_box):
    # since yolo normalized, box coords in [0,1]. Compute fraction inside image (simple)
    cx,cy,w,h = yolo_box
    x1 = cx - w/2; y1 = cy - h/2; x2 = cx + w/2; y2 = cy + h/2
    x1c, y1c = max(0.0, x1), max(0.0, y1)
    x2c, y2c = min(1.0, x2), min(1.0, y2)
    inter_area = max(0.0, x2c - x1c) * max(0.0, y2c - y1c)
    orig_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if orig_area <= 0:
        return 0.0
    return inter_area / orig_area

def save_sample(img, boxes, classes, out_img_path, out_lbl_path):
    cv2.imwrite(out_img_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    with open(out_lbl_path, "w") as f:
        for c, b in zip(classes, boxes):
            # clip numeric values and write trimmed
            b_clipped = [min(max(float(x), 0.0), 1.0) for x in b]
            f.write(f"{int(c)} {' '.join([f'{val:.6f}' for val in b_clipped])}\n")

# --- Build combined list (you already produced) ---
all_gold = glob(f"{GOLD_DIR}/images/*.png")
all_pseudo = glob(f"{PSEUDO_DIR}/images/*.png")
all_combined = all_gold + all_pseudo
random.shuffle(all_combined)

# split 80/20 (preserve labels)
split_idx = int(len(all_combined) * 0.8)
train_list = all_combined[:split_idx]
val_list   = all_combined[split_idx:]

for split in ['train', 'val']:
    os.makedirs(f"{FINAL_DIR}/{split}/images", exist_ok=True)
    os.makedirs(f"{FINAL_DIR}/{split}/labels", exist_ok=True)


# copy val as-is (you likely did this before; still ensure)
for img_path in tqdm(val_list, desc="Copying VAL"):
    name = os.path.basename(img_path)
    lbl = (GOLD_DIR if GOLD_DIR in img_path else PSEUDO_DIR) + "/labels/" + name.replace(".png", ".txt")
    if os.path.exists(lbl):
        shutil.copy(img_path, os.path.join(FINAL_DIR, "val/images", name))
        shutil.copy(lbl, os.path.join(FINAL_DIR, "val/labels", name.replace(".png", ".txt")))

# Prepare counts and estimate current train size
os.makedirs(os.path.join(FINAL_DIR, "train/images"), exist_ok=True)
os.makedirs(os.path.join(FINAL_DIR, "train/labels"), exist_ok=True)

# Copy originals into train (optionally)
if KEEP_ORIGINAL:
    for img_path in tqdm(train_list, desc="Copying original TRAIN"):
        name = os.path.basename(img_path)
        lbl = (GOLD_DIR if GOLD_DIR in img_path else PSEUDO_DIR) + "/labels/" + name.replace(".png", ".txt")
        if os.path.exists(lbl):
            shutil.copy(img_path, os.path.join(FINAL_DIR, "train/images", name))
            shutil.copy(lbl, os.path.join(FINAL_DIR, "train/labels", name.replace(".png", ".txt")))

# Count current train images
current_train = len(glob(os.path.join(FINAL_DIR, "train/images", "*.*")))
remaining_to_gen = max(0, TARGET_TOTAL_TRAIN - current_train)
print(f"Current train images: {current_train}, need to generate: {remaining_to_gen}")

# --- Augmentation loop with safety checks ---
generated = 0
pbar = tqdm(train_list, desc="Augmenting train (safety)", total=len(train_list))
for img_path in pbar:
    if generated >= remaining_to_gen:
        break

    name = os.path.basename(img_path)
    stem = Path(name).stem
    lbl_path = (GOLD_DIR if GOLD_DIR in img_path else PSEUDO_DIR) + "/labels/" + name.replace(".png", ".txt")
    if not os.path.exists(lbl_path):
        continue

    # read image and labels
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        continue
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    orig_boxes, orig_classes = read_yolo_label(lbl_path)
    if not orig_boxes:
        continue

    # compute original box areas for later comparison
    orig_areas = [yolo_area(b) for b in orig_boxes]

    # decide how many aug to produce for this source
    # heuristics: generate more for rare classes in this image
    cnt_per_class = {}
    for c in set(orig_classes):
        cnt_per_class[c] = sum(1 for x in orig_classes if x==c)

    # target per image (adaptive): try to distribute remaining across images
    approx_left_images = max(1, len(train_list))
    target_per_image = min(MAX_AUG_PER_IMAGE, math.ceil(remaining_to_gen / approx_left_images))
    tries = 0
    local_generated = 0

    while local_generated < target_per_image and generated < remaining_to_gen and tries < MAX_TRIES_PER_IMAGE:
        tries += 1

        # apply safe transform
        try:
            out = safe_transform(image=img, bboxes=orig_boxes, class_labels=orig_classes)
        except Exception:
            continue

        aug_img = out['image']
        aug_bboxes = out['bboxes']
        aug_classes = out['class_labels']

        # basic sanity: same number of boxes
        if len(aug_bboxes) != len(orig_boxes):
            # albumentations might drop boxes that go fully outside -> reject
            continue

        # check each bbox: area ratio and inside fraction
        ok = True
        for idx, (ob, ab) in enumerate(zip(orig_boxes, aug_bboxes)):
            orig_area = orig_areas[idx]
            new_area = yolo_area(ab)
            if new_area < orig_area * MIN_AREA_RATIO:
                ok = False
                break
            inside_frac = bbox_inside_fraction(ab)
            if inside_frac < MIN_IOU_INSIDE:
                ok = False
                break

        if not ok:
            continue  # discard this augmentation, try again

        # passed checks: save augmented
        aug_tag = f"{stem}_aug_{generated}"
        out_img_path = os.path.join(FINAL_DIR, "train/images", aug_tag + ".png")
        out_lbl_path = os.path.join(FINAL_DIR, "train/labels", aug_tag + ".txt")
        save_sample(aug_img, aug_bboxes, aug_classes, out_img_path, out_lbl_path)

        local_generated += 1
        generated += 1
        pbar.set_postfix({"gen_total": generated})
    # end while per image

print(f"\n=== AUGMENTATION FINISHED ===\nGenerated: {generated} new images.")
print("Final train size:", len(glob(os.path.join(FINAL_DIR, "train/images", "*.*"))))


/tmp/ipython-input-3763293820.py:41: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 25.0), p=0.5),
/tmp/ipython-input-3763293820.py:54: UserWarning: Argument(s) 'shift_limit' are not valid for transform OpticalDistortion
  A.OpticalDistortion(distort_limit=0.02, shift_limit=0.01, p=0.15)
Copying original TRAIN: 100%|██████████| 543/543 [00:00<00:00, 6724.54it/s]


Current train images: 522, need to generate: 8478


Augmenting train (safety): 100%|██████████| 543/543 [03:51<00:00,  2.34it/s, gen_total=7840]



=== AUGMENTATION FINISHED ===
Generated: 7840 new images.
Final train size: 8362


In [ ]:
from google.colab import drive
import shutil
import os

# drive.mount("/content/drive", force_remount=True)

FINAL_DIR = "/content/final_dataset"
ZIP_PATH = "/content/drive/MyDrive/TechNo_Project/final_dataset_v1.zip"

# Nén toàn bộ dataset
shutil.make_archive("/content/final_dataset_v1", 'zip', FINAL_DIR)

# Copy sang Drive
shutil.copy("/content/final_dataset_v1.zip", ZIP_PATH)

print("✅ Dataset đã được đóng gói và lưu vào Drive!")
print("📦 File:", ZIP_PATH)


✅ Dataset đã được đóng gói và lưu vào Drive!
📦 File: /content/drive/MyDrive/TechNo_Project/final_dataset_v1.zip


In [ ]:
# from google.colab import drive
# import shutil
# import os

# drive.mount('/content/drive')

# ZIP_PATH = "/content/drive/MyDrive/TechNo_Project/final_dataset_v1.zip"
# EXTRACT_TO = "/content/"

# shutil.unpack_archive(ZIP_PATH, EXTRACT_TO)

# print("✅ Dataset đã sẵn sàng tại /content/final_dataset")


# SEED 2

In [ ]:
yaml_content = """
path: /content/final_dataset
train: train/images
val: val/images

names:
  0: HSA
  1: RHSA
  2: DT
  3: DB
"""
with open('final_data.yaml', 'w') as f:
    f.write(yaml_content)
print("Wrote final_data.yaml")


Wrote final_data.yaml


In [ ]:
# === TRAIN CELL (use this) ===
from ultralytics import YOLO
import torch, os, glob, shutil, time
from pathlib import Path

# --- User params (edit if muốn) ---
SAVE_TO_DRIVE = "/content/drive/MyDrive/TechNo_Project/Model_Final"   # nơi lưu trực tiếp vào Drive
RUN_NAME = "Final_v1"
PROJECT = SAVE_TO_DRIVE
EPOCHS = 150               # bạn có thể set cao (Colab Pro) — thay đổi nếu muốn
IMGSZ = 640
MODEL_BACKBONE = "yolov8n.pt"   # nhanh, đủ cho task
AMP = True
CACHE = "ram"   # 'ram' or 'disk' or False. ram fastest if memory permits.
SAVE_PERIOD = 3  # save checkpoint every epoch
BATCH = 32  # start value; if OOM reduce
DEVICE = 0  # GPU id; use -1 for CPU

# create save dir on Drive
os.makedirs(PROJECT, exist_ok=True)

# detect GPU and adjust batch size if needed (heuristic)
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(DEVICE)
    # Tesla T4 ~16GB -> batch 32 reasonable for imgsz=640 on small model, adjust down if OOM
    print("GPU:", dev.name, "| total mem (GB):", dev.total_memory/1e9)
else:
    print("No GPU detected, training will use CPU (very slow)")

# --- find if previous checkpoint exists for RESUME ---
# ultralytics typically writes runs to PROJECT/<name>/weights/*.pt (best.pt, last.pt, epochXX.pt)
run_weights_dir = os.path.join(PROJECT, RUN_NAME, "weights")
last_ckpt = None
if os.path.isdir(run_weights_dir):
    # prefer last.pt, otherwise best.pt or latest epoch file
    cand = glob.glob(os.path.join(run_weights_dir, "last*.pt")) + glob.glob(os.path.join(run_weights_dir, "last.pt"))
    if not cand:
        cand = glob.glob(os.path.join(run_weights_dir, "best.pt")) + glob.glob(os.path.join(run_weights_dir, "epoch*.pt"))
    if cand:
        # pick newest by mtime
        last_ckpt = sorted(cand, key=os.path.getmtime)[-1]
        print("Found checkpoint for resume:", last_ckpt)

# --- instantiate model ---
model = YOLO(MODEL_BACKBONE)

# --- training call ---
train_kwargs = dict(
    data = "final_data.yaml",
    epochs = EPOCHS,
    imgsz = IMGSZ,
    batch = BATCH,
    project = PROJECT,
    name = RUN_NAME,
    device = DEVICE,
    cache = CACHE,
    amp = AMP,
    save = True,
    save_period = SAVE_PERIOD,   # save every epoch
    patience = 50,   # early stop patience, optional
)

# If checkpoint found, resume
if last_ckpt:
    print("Resuming training using checkpoint:", last_ckpt)
    # two ways: either point model to checkpoint weights first, or pass resume=True
    # Approach: load model weights then call train with resume=True
    model = YOLO(last_ckpt)   # load last weights
    # when resume=True, ultralytics attempts to continue training with existing run args
    train_kwargs['resume'] = True
else:
    print("No checkpoint found, starting fresh training.")

# Start training
print("Training started with args:", train_kwargs)
model.train(**train_kwargs)
print("Training finished.")


GPU: Tesla T4 | total mem (GB): 15.828320256
No checkpoint found, starting fresh training.
Training started with args: {'data': 'final_data.yaml', 'epochs': 150, 'imgsz': 640, 'batch': 32, 'project': '/content/drive/MyDrive/TechNo_Project/Model_Final', 'name': 'Final_v1', 'device': 0, 'cache': 'ram', 'amp': True, 'save': True, 'save_period': 3, 'patience': 50}
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hs

# Recall Model

In [ ]:
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v1/weights/best.pt"
model = YOLO(MODEL_PATH)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [ ]:
from google.colab import drive
import shutil
import os


ZIP_PATH = "/content/drive/MyDrive/TechNo_Project/final_dataset_v1.zip"
EXTRACT_TO = "/content/"

shutil.unpack_archive(ZIP_PATH, EXTRACT_TO)

print("✅ Dataset đã sẵn sàng tại /content")


✅ Dataset đã sẵn sàng tại /content/final_dataset


In [ ]:
from google.colab import drive
import shutil
import os

# drive.mount("/content/drive", force_remount=True)

FINAL_DIR = "/content/final_dataset"
ZIP_PATH = "/content/drive/MyDrive/TechNo_Project/final_dataset_v1.zip"

# Nén toàn bộ dataset
shutil.make_archive("/content/final_dataset_v1", 'zip', FINAL_DIR)

# Copy sang Drive
shutil.copy("/content/final_dataset_v1.zip", ZIP_PATH)

print("✅ Dataset đã được đóng gói và lưu vào Drive!")
print("📦 File:", ZIP_PATH)


In [ ]:
yaml_content = """
path: /content
train: train/images
val: val/images

names:
  0: HSA
  1: RHSA
  2: DT
  3: DB
"""
with open('final_data.yaml', 'w') as f:
    f.write(yaml_content)
print("Wrote final_data.yaml")

Wrote final_data.yaml


In [ ]:
metrics = model.val(data='final_data.yaml', imgsz=640)


Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 260.8±103.0 MB/s, size: 6.4 KB)
val: Scanning /content/val/labels... 132 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 132/132 1.2Kit/s 0.1s
val: New cache created: /content/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 3.5it/s 2.6s
                   all        132        132      0.687      0.776      0.813      0.679
                   HSA         72         72      0.755      0.986      0.977      0.872
                  RHSA         31         31      0.628      0.935      0.908      0.855
                    DT         10         10      0.753        0.5      0.609       0.47
                    DB         19         19       0.61      0.684      0.756      0.519
Speed: 2.1ms preprocess, 4.9ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /c

In [ ]:
import os
from ultralytics import YOLO
from glob import glob
import shutil
from tqdm import tqdm

# 1. Load model best.pt vừa train xong
model = YOLO('/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v1/weights/best.pt')

RAW_IMAGES = "/content/raw_images"
BOOSTER_DIR = "/content/booster_data"
os.makedirs(f"{BOOSTER_DIR}/images", exist_ok=True)
os.makedirs(f"{BOOSTER_DIR}/labels", exist_ok=True)

raw_files = glob(f"{RAW_IMAGES}/**/*.png", recursive=True)

print("🎯 Đang săn lùng riêng Double Top (DT) và Double Bottom (DB)...")
for img_p in tqdm(raw_files):
    # Hạ conf xuống 0.5 để bắt được nhiều mẫu DT hơn (vì model đang yếu lớp này)
    results = model.predict(img_p, conf=0.6, verbose=False)

    for box in results[0].boxes:
        cls = int(box.cls[0])
        # Chỉ lấy class 2 (DT) và 3 (DB)
        if cls in [2, 3]:
            name = os.path.basename(img_p)
            shutil.copy(img_p, f"{BOOSTER_DIR}/images/{name}")
            results[0].save_txt(f"{BOOSTER_DIR}/labels/{name.replace('.png', '.txt')}")
            break

print(f"✅ Đã tìm thêm được {len(os.listdir(BOOSTER_DIR+'/images'))} mẫu tập trung vào DT/DB!")

🎯 Đang săn lùng riêng Double Top (DT) và Double Bottom (DB)...


100%|██████████| 4664/4664 [00:54<00:00, 86.02it/s]

✅ Đã tìm thêm được 18 mẫu tập trung vào DT/DB!


In [ ]:
# import shutil

# shutil.rmtree("/content/booster_data", ignore_errors=True)


In [ ]:
import os
import random
import shutil
from glob import glob

# Đường dẫn nguồn (18 mẫu mới)
BOOSTER_IMG = "/content/booster_data/images"
BOOSTER_LBL = "/content/booster_data/labels"

# Đường dẫn đích (Thư mục bạn đã unzip)
TRAIN_DIR = "/content/train"
VAL_DIR = "/content/val"

# Lấy danh sách 18 file
all_files = [f for f in os.listdir(BOOSTER_IMG) if f.endswith('.png')]
random.shuffle(all_files)

# Chia 13 Val, 5 Train
val_files = all_files[:13]
train_files = all_files[13:]

print(f"✅ Đã chọn {len(val_files)} mẫu cho Validation và {len(train_files)} mẫu cho Training.")

✅ Đã chọn 13 mẫu cho Validation và 5 mẫu cho Training.


In [ ]:
print("🚚 Đang chuyển 13 mẫu vào tập Validation...")

for file_name in val_files:
    # Copy ảnh
    shutil.copy(os.path.join(BOOSTER_IMG, file_name), os.path.join(VAL_DIR, "images", file_name))

    # Copy nhãn
    label_name = file_name.replace('.png', '.txt')
    if os.path.exists(os.path.join(BOOSTER_LBL, label_name)):
        shutil.copy(os.path.join(BOOSTER_LBL, label_name), os.path.join(VAL_DIR, "labels", label_name))

print(f"✅ Hoàn tất! Tập Val hiện tại có: {len(os.listdir(VAL_DIR+'/images'))} ảnh.")

🚚 Đang chuyển 13 mẫu vào tập Validation...
✅ Hoàn tất! Tập Val hiện tại có: 145 ảnh.


In [ ]:
import albumentations as A
import cv2
from tqdm import tqdm

# Định nghĩa bộ tăng cường (Không Flip, không Crop mất gốc)
booster_transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.8),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.6),
    A.Blur(blur_limit=3, p=0.4),
    # Zoom in nhẹ (scale > 1.0), không xoay, không dịch chuyển
    A.ShiftScaleRotate(shift_limit=0, rotate_limit=0, scale_limit=(0.0, 0.2), p=0.5)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print(f"🔥 Đang tăng cường 5 mẫu mới (x50 lần)...")

for file_name in tqdm(train_files):
    img_path = os.path.join(BOOSTER_IMG, file_name)
    lbl_path = os.path.join(BOOSTER_LBL, file_name.replace('.png', '.txt'))

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with open(lbl_path, 'r') as f:
        bboxes = [list(map(float, line.split())) for line in f.readlines()]

    # 1. Lưu ảnh gốc vào Train
    shutil.copy(img_path, os.path.join(TRAIN_DIR, "images", file_name))
    shutil.copy(lbl_path, os.path.join(TRAIN_DIR, "labels", file_name.replace('.png', '.txt')))

    # 2. Tạo 50 bản sao tăng cường
    for i in range(50):
        cls_ids = [int(x[0]) for x in bboxes]
        coords = [x[1:] for x in bboxes]

        try:
            augmented = booster_transform(image=image, bboxes=coords, class_labels=cls_ids)
            aug_name = f"boost_{i}_{file_name}"

            # Lưu ảnh
            cv2.imwrite(os.path.join(TRAIN_DIR, "images", aug_name),
                        cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))

            # Lưu nhãn
            with open(os.path.join(TRAIN_DIR, "labels", aug_name.replace('.png', '.txt')), 'w') as f:
                for c, b in zip(augmented['class_labels'], augmented['bboxes']):
                    f.write(f"{c} {' '.join(map(str, b))}\n")
        except:
            continue

print(f"✅ Xong! Tập Train đã được bơm thêm dữ liệu DT/DB mới.")

/tmp/ipython-input-2669791936.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.6),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


🔥 Đang tăng cường 5 mẫu mới (x50 lần)...


100%|██████████| 5/5 [00:03<00:00,  1.50it/s]

✅ Xong! Tập Train đã được bơm thêm dữ liệu DT/DB mới.


In [ ]:
yaml_content = """
path: /content
train: train/images
val: val/images

names:
  0: HSA
  1: RHSA
  2: DT
  3: DB
"""
with open('final_data.yaml', 'w') as f:
    f.write(yaml_content)
print("Wrote final_data.yaml")

Wrote final_data.yaml


In [ ]:
# === FINE-TUNE CELL (Optimized for Colab Pro) ===
from ultralytics import YOLO
import torch, os, glob

# --- 1. Cấu hình đường dẫn ---
# Load model tốt nhất (v1 hoặc v2) để làm nền tảng
PREVIOUS_BEST = "/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v1/weights/best.pt"
SAVE_TO_DRIVE = "/content/drive/MyDrive/TechNo_Project/Model_Final"
RUN_NAME = "Final_v3_Refined_DT_DB"
PROJECT = SAVE_TO_DRIVE

# --- 2. Tham số tối ưu cho Colab Pro (High RAM/High GPU) ---
# Vì là Fine-tuning, chúng ta hạ lr0 (tốc độ học) xuống để tránh làm mất kiến thức HSA
train_kwargs = dict(
    model=PREVIOUS_BEST,    # Bắt đầu từ kiến thức của model 81.1%
    data="final_data.yaml",
    epochs=100,             # Fine-tune không cần quá dài, 100 là đủ
    imgsz=640,
    batch=32,               # Colab Pro (A100/L4) dư sức chạy batch 32-64
    project=PROJECT,
    name=RUN_NAME,
    device=0,
    cache="ram",            # Tận dụng High RAM của Pro để nạp dữ liệu cực nhanh
    amp=True,
    patience=30,            # Dừng sớm nếu DT/DB đã đạt đỉnh
    save_period=5,

    # --- THÔNG SỐ VÀNG CHO FINE-TUNING ---
    lr0=0.0005,             # Tốc độ học thấp (mặc định là 0.01) để học từ từ
    lrf=0.01,               # Final LR factor
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    # --- Tăng cường độ khó để máy học kỹ hơn ---
    mixup=0.1,              # Giúp máy không bị quá tự tin (overconfident)
    mosaic=1.0,             # Ép máy tìm mẫu hình trong các bối cảnh phức tạp
    close_mosaic=10,        # Tắt mosaic ở 10 epoch cuối để ổn định trọng số
)

# --- 3. Khởi tạo và Kiểm tra ---
os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists(PREVIOUS_BEST):
    print(f"⚠️ CẢNH BÁO: Không tìm thấy file {PREVIOUS_BEST}")
    print("Vui lòng kiểm tra lại đường dẫn file best.pt của Model v1!")
else:
    print(f"✅ Đang tải 'Chuyên gia v1' từ: {PREVIOUS_BEST}")
    model = YOLO(PREVIOUS_BEST)

    # Kiểm tra GPU Pro
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🔥 Đang sử dụng GPU cực mạnh: {gpu_name}")

    # --- 4. Thực thi Training ---
    print(f"🚀 Bắt đầu quá trình Tinh chỉnh (Fine-tuning) cho DT và DB...")
    model.train(**train_kwargs)
    print("✅ Quá trình học chuyển giao hoàn tất!")

✅ Đang tải 'Chuyên gia v1' từ: /content/drive/MyDrive/TechNo_Project/Model_Final/Final_v1/weights/best.pt
🔥 Đang sử dụng GPU cực mạnh: Tesla T4
🚀 Bắt đầu quá trình Tinh chỉnh (Fine-tuning) cho DT và DB...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=t

#YOLOv8n


In [ ]:
import os
from ultralytics import YOLO
from glob import glob
import shutil
from tqdm import tqdm

# Load model
model = YOLO('/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v1/weights/best.pt')

RAW_IMAGES = "/content/raw_images"
BOOSTER_DIR = "/content/booster_data_DT"

os.makedirs(f"{BOOSTER_DIR}/images", exist_ok=True)
os.makedirs(f"{BOOSTER_DIR}/labels", exist_ok=True)

raw_files = glob(f"{RAW_IMAGES}/**/*.png", recursive=True)

print("🎯 Đang săn riêng Double Top (DT)...")

count = 0

for img_p in tqdm(raw_files):
    results = model.predict(img_p, conf=0.35, verbose=False)[0]

    dt_boxes = []

    for box in results.boxes:
        cls = int(box.cls[0])

        # CHỈ lấy DT
        if cls == 2:
            dt_boxes.append(box)

    if len(dt_boxes) > 0:
        name = os.path.basename(img_p)

        # Copy image
        shutil.copy(img_p, f"{BOOSTER_DIR}/images/{name}")

        # Tạo label chỉ chứa DT
        label_path = f"{BOOSTER_DIR}/labels/{name.replace('.png', '.txt')}"

        with open(label_path, "w") as f:
            for box in dt_boxes:
                cls = int(box.cls[0])
                xywh = box.xywhn[0].tolist()
                f.write(f"{cls} {' '.join(map(str, xywh))}\n")

        count += 1

print(f"✅ Tổng ảnh nghi ngờ chứa DT để bạn manual check: {count}")


🎯 Đang săn riêng Double Top (DT)...


100%|██████████| 4664/4664 [00:58<00:00, 80.26it/s]

✅ Tổng ảnh nghi ngờ chứa DT để bạn manual check: 22


In [ ]:
# import shutil

# shutil.rmtree("/content/booster_data_DT", ignore_errors=True)

In [ ]:
import os
import random
import shutil
from glob import glob

# Giả sử 22 mẫu mới đang nằm ở đây (bạn điều chỉnh nếu folder khác)
SOURCE_IMG = "/content/booster_data_DT/images"
SOURCE_LBL = "/content/booster_data_DT/labels"

TRAIN_DIR = "/content/train"
VAL_DIR = "/content/val"

# Lấy danh sách 22 file
all_files = [f for f in os.listdir(SOURCE_IMG) if f.endswith('.png')]
random.seed(42) # Giữ kết quả cố định
random.shuffle(all_files)

val_files = all_files[:18]
train_files = all_files[18:]

print(f"📦 Chia thành công: {len(val_files)} vào Val, {len(train_files)} vào Train.")

# Chuyển 18 mẫu vào Val
for f in val_files:
    shutil.copy(os.path.join(SOURCE_IMG, f), os.path.join(VAL_DIR, "images", f))
    lbl = f.replace('.png', '.txt')
    shutil.copy(os.path.join(SOURCE_LBL, lbl), os.path.join(VAL_DIR, "labels", lbl))

print("✅ Đã cập nhật tập Validation.")

📦 Chia thành công: 18 vào Val, 4 vào Train.
✅ Đã cập nhật tập Validation.


In [ ]:
import albumentations as A
import cv2
from tqdm import tqdm

# Bộ tăng cường bảo tồn cấu trúc (Không Flip, không Crop)
dt_extreme_aug = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.9),
    A.GaussNoise(var_limit=(20, 100), p=0.8),
    A.Blur(blur_limit=3, p=0.5),
    A.ShiftScaleRotate(shift_limit=0, rotate_limit=0, scale_limit=(0.0, 0.15), p=0.7) # Zoom in nhẹ
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

print(f"🔥 Đang tạo 600 biến thể từ 4 mẫu DT...")

for file_name in tqdm(train_files):
    img_path = os.path.join(SOURCE_IMG, file_name)
    lbl_path = os.path.join(SOURCE_LBL, file_name.replace('.png', '.txt'))

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    with open(lbl_path, 'r') as f:
        bboxes = [list(map(float, line.split())) for line in f.readlines()]

    # Tạo 40 bản sao mỗi ảnh
    for i in range(40):
        cls_ids = [int(x[0]) for x in bboxes]
        coords = [x[1:] for x in bboxes]

        try:
            augmented = dt_extreme_aug(image=image, bboxes=coords, class_labels=cls_ids)
            aug_name = f"dt_special_{i}_{file_name}"

            cv2.imwrite(os.path.join(TRAIN_DIR, "images", aug_name),
                        cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))

            with open(os.path.join(TRAIN_DIR, "labels", aug_name.replace('.png', '.txt')), 'w') as f:
                for c, b in zip(augmented['class_labels'], augmented['bboxes']):
                    f.write(f"{c} {' '.join(map(str, b))}\n")
        except:
            continue

print(f"✅ Đã bơm thêm 160 mẫu DT vào tập Train.")

/tmp/ipython-input-200722573.py:8: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(20, 100), p=0.8),
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


🔥 Đang tạo 600 biến thể từ 4 mẫu DT...


100%|██████████| 4/4 [00:02<00:00,  1.49it/s]

✅ Đã bơm thêm 160 mẫu DT vào tập Train.


In [ ]:
yaml_content = """
path: /content
train: train/images
val: val/images

names:
  0: HSA
  1: RHSA
  2: DT
  3: DB
"""
with open('final_data.yaml', 'w') as f:
    f.write(yaml_content)
print("Wrote final_data.yaml")

Wrote final_data.yaml


In [ ]:
from ultralytics import YOLO

# Load model tốt nhất từ đợt v3 vừa rồi
PREVIOUS_BEST = "/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v3_Refined_DT_DB/weights/best.pt"
model = YOLO(PREVIOUS_BEST)

model.train(
    data="final_data.yaml",
    epochs=100,            # Tăng lên 100 vì lr rất thấp
    imgsz=640,
    batch=32,              # Tối ưu cho Colab Pro
    lr0=0.0001,           # Tốc độ học RẤT THẤP để bảo toàn HSA/RHSA
    lrf=0.01,
    project="/content/drive/MyDrive/TechNo_Project/Model_Final",
    name="Final_v4_DT_Extreme",
    cache="ram",
    patience=50,

    # --- THIẾT LẬP CHIẾN THUẬT CHO LỚP YẾU ---
    cls=1.5,               # Tăng mạnh trọng số Class (mặc định 0.5) để sửa lỗi nhầm DT
    box=7.5,               # Giữ nguyên trọng số khung
    warmup_epochs=5,
    # mixup=0.15,            # Giúp model phân biệt các đỉnh nến chồng chéo tốt hơn
    overlap_mask=False
)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/TechNo_Project/Model_Final/Final_v3_Refined_DT_DB/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Final_v4_DT_Extreme, nbs=64, nms=False, opset=None, o

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7eefed606bd0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
import os
import zipfile
import shutil

ZIP_NAME = "/content/train_val_backup.zip"
DRIVE_DEST = "/content/drive/MyDrive/TechNo_Project/train_val_backup.zip"

print("📦 Đang nén /content/train và /content/val ...")

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ["train", "val"]:
        folder_path = f"/content/{folder}"
        for root, _, files in os.walk(folder_path):
            for file in files:
                full_path = os.path.join(root, file)
                # giữ nguyên cấu trúc thư mục khi nén
                z.write(full_path, os.path.relpath(full_path, "/content"))

print("✅ Nén xong:", ZIP_NAME)

# copy sang Drive
shutil.copy(ZIP_NAME, DRIVE_DEST)
print("✅ Đã lưu vào Drive:", DRIVE_DEST)


📦 Đang nén /content/train và /content/val ...
✅ Nén xong: /content/train_val_backup.zip
✅ Đã lưu vào Drive: /content/drive/MyDrive/TechNo_Project/train_val_backup.zip


#YOLOv10n

In [ ]:
# Cài đặt YOLOv10
!pip install -q git+https://github.com/THU-MIG/yolov10.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from ultralytics import YOLO # Vẫn dùng class YOLO nhưng nạp model v10
import torch, os

# --- 1. Cấu hình đường dẫn ---
SAVE_TO_DRIVE = "/content/drive/MyDrive/TechNo_Project/Model_Final"
RUN_NAME = "Final_YOLOv10_Comparison"
PROJECT = SAVE_TO_DRIVE

# --- 2. Tải file trọng số YOLOv10 Nano ---
# Nếu file chưa có, nó sẽ tự động tải về
model_path = "yolov10n.pt"
if not os.path.exists(model_path):
    !wget https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt

model = YOLO(model_path)

# --- 3. Tham số Training (Giữ nguyên để so sánh công bằng) ---
train_kwargs = dict(
    data="final_data.yaml",
    epochs=150,
    imgsz=640,
    batch=32,
    project=PROJECT,
    name=RUN_NAME,
    device=0,
    cache="ram",
    amp=True,
    patience=50,
    save_period=5,
)

print(f"🚀 Bắt đầu huấn luyện YOLOv10 (NMS-Free architecture)...")
model.train(**train_kwargs)

--2026-02-11 10:20:43--  https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/804788522/411e0d4f-1023-40ad-bfdd-c99f0dddb73b?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-11T11%3A15%3A29Z&rscd=attachment%3B+filename%3Dyolov10n.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-11T10%3A14%3A31Z&ske=2026-02-11T11%3A15%3A29Z&sks=b&skv=2018-11-09&sig=n8gNbw%2BLNnzAY7cHh3vfkTUrKHXFXGnxJaGduc952zA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MDgwNzA0MywibmJmIjoxNzcwODA1MjQzLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcm

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ef1ce7ebbf0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

# YOLOv11n

In [ ]:
from ultralytics import YOLO

# 1. Khởi tạo mô hình v11 nano (yolo11n)
model = YOLO("yolo11n.pt")

# 2. Train (Dùng đúng bộ tham số tối ưu bạn đã tìm ra ở v8)
train_kwargs = dict(
    data="final_data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    project="/content/drive/MyDrive/TechNo_Project/Model_Final",
    name="Final_YOLOv11_Comparison",
    device=0,
    cache="ram",
    amp=True,
    patience=30
)

model.train(**train_kwargs)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Final_YOLOv11_Comparison, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspective=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ef0665057f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

#YOLOv26n

In [ ]:
from ultralytics import YOLO
import torch

# 1. Khởi tạo YOLO26 Nano (phiên bản nhanh và nhẹ nhất)
model = YOLO("yolo26n.pt")

# 2. Cấu hình Training
train_kwargs = dict(
    data="final_data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    project="/content/drive/MyDrive/TechNo_Project/Model_Final",
    name="Final_YOLO26_Comparison",
    device=0,
    cache="ram",
    amp=True,
    patience=50,
    # Giữ nguyên các tham số để so sánh công bằng với v8
)

print("🚀 Khởi động 'Quái thú' YOLO26...")
model.train(**train_kwargs)

🚀 Khởi động 'Quái thú' YOLO26...
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=final_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Final_YOLO26_Comparison, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ef04c128410>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0